In [1]:
import torch
import matplotlib.pyplot as plt
import os
import time
import numpy as np
import random
from matplotlib.patches import Rectangle
import h5py
from torch.utils.data import DataLoader

import importlib as imp
import sys
sys.path.append('/home/abenneck/Desktop/yolo_tiles/docs/scripts')

import yolo_tiles
imp.reload(yolo_tiles)
from yolo_tiles import img_to_tiles, apply_model_to_tiles, load_test_image, preprocess, tileDataset, remove_bbox_in_overlap, tiles_to_orig

import yolo_help
imp.reload(yolo_help)
from yolo_help import bbox_to_rectangles, imshow, convert_data, Net, get_best_bounding_box_per_cell

import yolo_post_help
imp.reload(yolo_post_help)
from yolo_post_help import remove_low_conf_bboxes, postprocess, bb_to_rec

In [2]:
start_total = time.time()

# Laod 2D YOLO model
model_path = '/nafs/shattuck/RodentToolsData/ZW-DT-1-P56-1/yolo_saved_weights/modelsave_bright_on_dark.pt'
net = Net()
net.load_state_dict(torch.load(model_path))
B = net.B
stride = net.stride

# Load HDF5 3D input image
# fname = '/nafs/shattuck/RodentToolsData/Stacked/Ex_488_Em_525_stitched_chunk32.h5'
fname = '/nafs/shattuck/RodentToolsData/Stacked/Ex_488_Em_525_stitched_full_sag.h5'
vol = h5py.File(fname)
vol_stack = vol['stack']
curr_ax = 2 # [0,1,2]
next_idx = 7321

# outdir = f'/home/abenneck/Desktop/yolo3d_outputs/Ex_488_Em_525/ax{curr_ax}'
outdir = f'/nafs/shattuck/RodentToolsData/ZW-DT-1-P56-1/yolo3D_outputs_v01/ax{curr_ax}'

for idx in np.arange(vol_stack.shape[curr_ax]):

    if idx < next_idx:
        continue

    # Extract the slice 
    if curr_ax == 0:
        img = vol_stack[idx,:,:]
        out_fname = os.path.split(fname)[-1][:-3] + f'idx_{idx:05d}_:_:.npy'
    elif curr_ax == 1:
        img = vol_stack[:,idx,:]
        out_fname = os.path.split(fname)[-1][:-3] + f'idx_:_{idx:05d}_:.npy'
    elif curr_ax == 2:
        img = vol_stack[:,:,idx]
        out_fname = os.path.split(fname)[-1][:-3] + f'idx_:_:_{idx:05d}.npy'
    else:
        raise Exception(f'Invalid ax ({curr_ax}) supplied, only ({[k for k in range(2)]}) allowed')

    # Preprocess using gamma correction + upsampling
    start = time.time()
    img_up = preprocess(img)
    print(f'Finished preprocessing in {time.time()-start:.2f}s')
    
    # Extract tiles from the preprocessed input image
    padded_img, tiles = img_to_tiles(img_up, lower_threshold_bg = 0.04, verbose=True)
    
    # Apply model to tiles + apply bbox edge filtering
    print('Applying model to tiles . . .')
    out = apply_model_to_tiles(tiles, model_path, padded_img.shape[0], padded_img.shape[1], verbose=True)

    # Convert the raw model output into a more useful data structure
    print('Postprocessing . . .')
    pads = (np.array(padded_img.shape) - np.array(img_up.shape))/2
    out = torch.tensor(out.clone().detach(), dtype=torch.float32)
    out = postprocess(out, B, stride, pads, up_factor=2, verbose=True)

    # Save the processed output
    out_path = os.path.join(outdir, out_fname)
    if True:
        np.save(out_path, out)

    # # (11/19/25) Save some additional metadata in an npz file
    # out_fnamez = (img_path.split('/')[-1]).split('.')[0] + '_metadata.npz'
    # out_pathz = os.path.join(outdir, out_fnamez)
    # if False:
    #     np.savez(out_pathz,
    #             units = 'um',
    #             nrow_orig_img = img.shape[0],
    #             ncol_orig_img = img.shape[1],
    #             drow_orig_img = 1.866,
    #             dcol_orig_img = 1.866,
    #             dslice_orig_img = 2.0,
    #             nrow_upsampled_img = img_up.shape[0],
    #             ncol_upsampled_img = img_up.shape[1], 
    #             drow_upsampled_img = 3.732,
    #             dcol_upsampled_img = 3.732,
    #             pad_row_top = pads[0],
    #             pad_row_bot = pads[0],
    #             pad_col_left = pads[1],
    #             pad_col_right = pads[1],
    #             nrow = out.shape[0],
    #             ncol = out.shape[1],
    #             drow = 7.464,
    #             dcol = 7.464,
    #             note = "Bbox coords refer to the original image, with (0,0) corresponding to the top left pixel center"
    #             )    
        
    print(f'Saved the outputs for image {idx}/{vol_stack.shape[curr_ax]} in {time.time()-start_total:.2f}s\n')
    start_total = time.time()



Finished preprocessing in 1.49s
Finished extracting all tiles in 3.98s
Applying model to tiles . . .
Finished tiles 0:200/2541 in 2.36s
Finished tiles 200:300/2541 in 2.37s
Finished tiles 300:400/2541 in 2.33s
Finished tiles 400:500/2541 in 2.38s
Finished tiles 500:600/2541 in 2.53s
Finished tiles 600:700/2541 in 2.60s
Finished tiles 700:800/2541 in 2.57s
Finished tiles 800:900/2541 in 2.43s
Finished tiles 900:1100/2541 in 4.99s
Finished tiles 1100:1200/2541 in 2.43s
Finished tiles 1200:1300/2541 in 2.40s
Finished tiles 1300:1400/2541 in 2.29s
Finished tiles 1400:1500/2541 in 2.68s
Finished tiles 1500:1600/2541 in 2.59s
Finished tiles 1600:1700/2541 in 2.33s
Finished tiles 1700:1800/2541 in 2.02s
Finished tiles 1800:2000/2541 in 4.27s
Finished tiles 2000:2100/2541 in 2.19s
Finished tiles 2100:2200/2541 in 2.28s
Finished tiles 2200:2300/2541 in 2.07s
Finished tiles 2300:2400/2541 in 2.42s
Finished applying model to entire image in 56.05s with 2222/2541 (0.874) tiles marked as foreground

/tmp/ipykernel_767235/415798440.py:54: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  out = torch.tensor(out.clone().detach(), dtype=torch.float32)


Saved the outputs for image 6768/7321 in 64.28s

Finished preprocessing in 1.66s
Finished extracting all tiles in 4.38s
Applying model to tiles . . .
Finished tiles 0:200/2541 in 2.48s
Finished tiles 200:300/2541 in 2.28s
Finished tiles 300:400/2541 in 2.33s
Finished tiles 400:500/2541 in 2.50s
Finished tiles 500:600/2541 in 2.52s
Finished tiles 600:700/2541 in 2.33s
Finished tiles 700:800/2541 in 2.30s
Finished tiles 800:900/2541 in 2.28s
Finished tiles 900:1100/2541 in 4.75s
Finished tiles 1100:1200/2541 in 2.47s
Finished tiles 1200:1300/2541 in 2.44s
Finished tiles 1300:1400/2541 in 2.38s
Finished tiles 1400:1500/2541 in 2.39s
Finished tiles 1500:1600/2541 in 2.30s
Finished tiles 1600:1700/2541 in 2.41s
Finished tiles 1700:1800/2541 in 2.32s
Finished tiles 1800:2000/2541 in 4.18s
Finished tiles 2000:2100/2541 in 2.19s
Finished tiles 2100:2200/2541 in 2.13s
Finished tiles 2200:2300/2541 in 2.06s
Finished tiles 2300:2400/2541 in 2.48s
Finished applying model to entire image in 55.20s 